# RAVE Dissertation Experiments
**Mihir Apte - MSc Data Science**

Before running:
1. Runtime > Change runtime type > A100 GPU + High RAM on
2. Run each cell in order (or Runtime > Run all)
3. Only Cell 7 needs manual action: uploading truck.mp4

In [ ]:
# Cell 1 - Check GPU
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print('VRAM    :', vram, 'GB')
else:
    raise RuntimeError('No GPU. Go to Runtime > Change runtime type and select A100.')

In [ ]:
# Cell 2 - Clone repo
import os
REPO = '/content/dissertation-mihir'
if os.path.exists(REPO):
    print('Repo exists, pulling latest...')
    os.system(f'cd {REPO} && git pull origin main')
else:
    os.system(f'git clone https://github.com/MihirApte/dissertation-mihir.git {REPO}')
os.chdir(REPO)
print('Working directory:', os.getcwd())

In [ ]:
# Cell 3 - Install packages (does NOT reinstall torch)
import subprocess
pkgs = [
    'diffusers==0.27.0 accelerate safetensors transformers',
    'basicsr timm==0.6.7 einops omegaconf ftfy regex imageio',
    'git+https://github.com/openai/CLIP.git',
    'mmdet==3.2.0 mmpose==1.2.0',
]
for pkg in pkgs:
    print(f'Installing {pkg.split()[0]}...')
    subprocess.run(f'pip install {pkg} -q', shell=True)
print('All packages installed.')

In [ ]:
# Cell 4 - Patch basicsr (torchvision removed functional_tensor in 0.17+)
import glob
matches = glob.glob('/usr/local/lib/python3.*/dist-packages/basicsr/data/degradations.py')
if matches:
    path = matches[0]
    with open(path) as f:
        content = f.read()
    if 'functional_tensor' in content:
        content = content.replace(
            'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
            'from torchvision.transforms.functional import rgb_to_grayscale'
        )
        with open(path, 'w') as f:
            f.write(content)
        print('basicsr patched OK')
    else:
        print('basicsr: no patch needed')
else:
    print('basicsr path not found - skipping')

In [ ]:
# Cell 5 - Patch ZoeDepth to run on CPU (saves GPU VRAM)
zoe_path = '/content/dissertation-mihir/annotator/zoe/__init__.py'
with open(zoe_path) as f:
    content = f.read()
content = content.replace(
    'self.model.to(self.device)',
    'self.model.to("cpu")'
).replace(
    'image_depth = torch.from_numpy(image_depth).float().to(self.device)',
    'image_depth = torch.from_numpy(image_depth).float().to("cpu")'
)
with open(zoe_path, 'w') as f:
    f.write(content)
print('ZoeDepth patched to CPU')

In [ ]:
# Cell 6 - Patch xformers (make optional so missing xformers does not crash)
pipe_path = '/content/dissertation-mihir/pipelines/sd_controlnet_rave.py'
with open(pipe_path) as f:
    content = f.read()
old = '        pipe.enable_model_cpu_offload()\n        pipe.enable_xformers_memory_efficient_attention()'
new = '        pipe.enable_model_cpu_offload()\n        try:\n            pipe.enable_xformers_memory_efficient_attention()\n        except ModuleNotFoundError:\n            pass'
if old in content:
    content = content.replace(old, new)
    with open(pipe_path, 'w') as f:
        f.write(content)
    print('xformers patch applied')
else:
    print('xformers already patched or not found - skipping')

In [ ]:
# Cell 7 - Upload truck.mp4
# ACTION NEEDED: click the upload button that appears below and select truck.mp4
import os, shutil
VIDEO = '/content/dissertation-mihir/data/mp4_videos/truck.mp4'
os.makedirs(os.path.dirname(VIDEO), exist_ok=True)
if os.path.exists(VIDEO):
    print('Video already present:', VIDEO)
else:
    from google.colab import files
    print('Select truck.mp4 from your computer...')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    shutil.move(fname, VIDEO)
    print('Video saved to:', VIDEO)

In [ ]:
# Cell 8 - Environment check (all 10 checks should pass)
import subprocess
result = subprocess.run(
    ['python', '/content/dissertation-mihir/check_gpu.py'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

In [ ]:
# Cell 9 - Run both experiments
# Baseline (random shuffle) then Semantic shuffle, then metrics
# Expected time: 30-60 min on A100, 60-90 min on L4
import os
os.chdir('/content/dissertation-mihir')
os.system('bash run_experiments.sh')

In [ ]:
# Cell 10 - Print results
results_file = '/content/dissertation-mihir/results/metrics_comparison.txt'
if os.path.exists(results_file):
    with open(results_file) as f:
        print(f.read())
else:
    print('Results file not found. Check experiment output above for errors.')

In [ ]:
# Cell 11 - Download results to your computer
import shutil
from google.colab import files
shutil.make_archive('/content/rave_results', 'zip', '/content/dissertation-mihir/results')
files.download('/content/rave_results.zip')
print('Downloading rave_results.zip...')